In [1]:
import pandas as pd
import numpy as np
from pathlib import Path




In [2]:
def read_csv_fallback(filename: str, **kwargs):
    """Read a CSV from /kaggle/input or a sub‑folder if needed."""
    base = Path("/kaggle/input")
    possible = [
        base / filename,
        base / "new-york-city-taxi-fare-prediction" / filename,
    ]
    for p in possible:
        if p.exists():
            return pd.read_csv(p, **kwargs)
    raise FileNotFoundError(f"{filename} not found in /kaggle/input")


train_dtype = {
    "key": "object",
    "fare_amount": "float32",
    "pickup_datetime": "object",
    "pickup_longitude": "float32",
    "pickup_latitude": "float32",
    "dropoff_longitude": "float32",
    "dropoff_latitude": "float32",
    "passenger_count": "int8",
}
test_dtype = {
    "key": "object",
    "pickup_datetime": "object",
    "pickup_longitude": "float32",
    "pickup_latitude": "float32",
    "dropoff_longitude": "float32",
    "dropoff_latitude": "float32",
    "passenger_count": "int8",
}

train_data = read_csv_fallback("train.csv", dtype=train_dtype, low_memory=False)
test_data = read_csv_fallback("test.csv", dtype=test_dtype, low_memory=False)




In [3]:
train_data.head()




,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,2009-06-15 17:26:21.0000001,4.5,2009-06-15 17:26:21 UTC,-73.844315,40.721317,-73.841614,40.712276,1
1,2010-01-05 16:52:16.0000002,16.9,2010-01-05 16:52:16 UTC,-74.016045,40.711304,-73.979271,40.782005,1
2,2011-08-18 00:35:00.00000049,5.7,2011-08-18 00:35:00 UTC,-73.982735,40.761269,-73.991241,40.750561,2
3,2012-04-21 04:30:42.0000001,7.7,2012-04-21 04:30:42 UTC,-73.987129,40.733143,-73.991570,40.758091,1
4,2010-03-09 07:51:00.000000135,5.3,2010-03-09 07:51:00 UTC,-73.968094,40.768009,-73.956657,40.783764,1


In [4]:
test_data.head()




,key,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,2010-10-01 21:26:11.0000001,2010-10-01 21:26:11 UTC,-73.983131,40.761971,-73.994385,40.749237,1
1,2013-10-06 01:38:00.00000083,2013-10-06 01:38:00 UTC,-73.948502,40.753979,-73.808197,40.731953,2
2,2012-03-30 19:13:53.0000001,2012-03-30 19:13:53 UTC,-73.973961,40.791981,-73.979019,40.785545,1
3,2012-02-08 02:57:23.0000001,2012-02-08 02:57:23 UTC,-73.991478,40.738907,-73.907196,40.861572,2
4,2013-12-13 22:56:00.000000237,2013-12-13 22:56:00 UTC,-73.986282,40.740067,-73.933929,40.856781,2


In [5]:
train_data["Difference_longitude"] = np.abs(
    train_data["pickup_longitude"] - train_data["dropoff_longitude"]
)
train_data["Difference_latitude"] = np.abs(
    train_data["pickup_latitude"] - train_data["dropoff_latitude"]
)

test_data["Difference_longitude"] = np.abs(
    test_data["pickup_longitude"] - test_data["dropoff_longitude"]
)
test_data["Difference_latitude"] = np.abs(
    test_data["pickup_latitude"] - test_data["dropoff_latitude"]
)




In [6]:
print(f"Before Dropping null values: {len(train_data)}")
train_data.dropna(inplace=True)
print(f"After Dropping null values: {len(train_data)}")
train_data = train_data[train_data["fare_amount"] > 0].reset_index(drop=True)
print(f"After removing non‑positive fares: {len(train_data)}")




Before Dropping null values: 55423856
After Dropping null values: 55423480
After removing non‑positive fares: 55419646


In [7]:
dt_train = pd.to_datetime(train_data["pickup_datetime"], errors="coerce")
dt_test = pd.to_datetime(test_data["pickup_datetime"], errors="coerce")

train_data = train_data[dt_train.notna()].reset_index(drop=True)
dt_train = dt_train[dt_train.notna()]

train_data["pickuptime"] = dt_train.dt.hour * 100 + dt_train.dt.minute
test_data["pickuptime"] = dt_test.dt.hour * 100 + dt_test.dt.minute

train_data["Weekday"] = dt_train.dt.weekday
test_data["Weekday"] = dt_test.dt.weekday

train_data.drop(columns=["pickup_datetime"], inplace=True)
test_data.drop(columns=["pickup_datetime"], inplace=True)




In [8]:
# One‑hot encode the weekday (categorical) feature for both train and test.
weekday_dummies_train = pd.get_dummies(train_data["Weekday"], prefix="wd")
weekday_dummies_test = pd.get_dummies(test_data["Weekday"], prefix="wd")

train_data = pd.concat(
    [train_data.drop(columns=["Weekday"]), weekday_dummies_train], axis=1
)
test_data = pd.concat(
    [test_data.drop(columns=["Weekday"]), weekday_dummies_test], axis=1
)




In [9]:
pass




In [10]:
pass




In [11]:
pass




In [12]:
pass




In [13]:
pass




In [14]:
R = 6373.0  # Earth radius in km


def haversine(df):
    lat1 = np.radians(df["pickup_latitude"])
    lon1 = np.radians(df["pickup_longitude"])
    lat2 = np.radians(df["dropoff_latitude"])
    lon2 = np.radians(df["dropoff_longitude"])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c * 0.621  # convert km → miles


train_data["Distance"] = haversine(train_data)
test_data["Distance"] = haversine(test_data)




In [15]:
airport_lat = np.radians(40.6413111)
airport_lon = np.radians(-73.7781391)


def airport_distance(lat_col, lon_col):
    lat = np.radians(lat_col)
    lon = np.radians(lon_col)
    dlon = airport_lon - lon
    dlat = airport_lat - lat
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat) * np.cos(airport_lat) * np.sin(dlon / 2) ** 2
    )
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c * 0.621


train_data["Pickup_Distance_airport"] = airport_distance(
    train_data["pickup_latitude"], train_data["pickup_longitude"]
)
train_data["Dropoff_Distance_airport"] = airport_distance(
    train_data["dropoff_latitude"], train_data["dropoff_longitude"]
)

test_data["Pickup_Distance_airport"] = airport_distance(
    test_data["pickup_latitude"], test_data["pickup_longitude"]
)
test_data["Dropoff_Distance_airport"] = airport_distance(
    test_data["dropoff_latitude"], test_data["dropoff_longitude"]
)




In [16]:
dist_cols = ["Distance", "Pickup_Distance_airport", "Dropoff_Distance_airport"]
for col in dist_cols:
    train_data[col] = np.round(train_data[col], 2)
    test_data[col] = np.round(test_data[col], 2)




In [17]:
sample_size = 2_000_000  # adjust as needed for the 600 s budget
if len(train_data) > sample_size:
    train_data = train_data.sample(n=sample_size, random_state=42).reset_index(
        drop=True
    )

num_cols = [
    "Difference_longitude",
    "Difference_latitude",
    "Distance",
    "Pickup_Distance_airport",
    "Dropoff_Distance_airport",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "pickuptime",
    "passenger_count",
]

stats = train_data[num_cols].agg(["mean", "std"]).T
train_data[num_cols] = (train_data[num_cols] - stats["mean"]) / stats["std"]
test_data[num_cols] = (test_data[num_cols] - stats["mean"]) / stats["std"]

for base in ["Distance", "passenger_count", "pickuptime"]:
    train_data[f"{base}_sq"] = train_data[base] ** 2
    test_data[f"{base}_sq"] = test_data[base] ** 2




In [18]:
train_features = set(train_data.columns) - {"key", "fare_amount"}
test_features = set(test_data.columns) - {"key"}

for col in train_features - test_features:
    test_data[col] = 0
for col in test_features - train_features:
    train_data[col] = 0




In [19]:
X = train_data.drop(columns=["key", "fare_amount"])
y = train_data["fare_amount"]
X_test = test_data.drop(columns=["key"])

X_array = X.values.astype(np.float32)
X_test_array = X_test.values.astype(np.float32)




In [20]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X_array, y, test_size=0.01, random_state=80
)




In [21]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

alpha_val = 0.0001  # slightly reduced regularisation
ridge_raw = Ridge(
    alpha=alpha_val,
    random_state=80,
    solver="sag",
    max_iter=1000,  # increased iterations for better convergence
    tol=1e-3,
    fit_intercept=True,
)
ridge_raw.fit(X_train, y_train)
pred_raw_val = ridge_raw.predict(X_val)
rmse_raw = mean_squared_error(y_val, pred_raw_val, squared=False)

ridge_log = Ridge(
    alpha=alpha_val,
    random_state=80,
    solver="sag",
    max_iter=1000,
    tol=1e-3,
    fit_intercept=True,
)
y_train_log = np.log1p(y_train.clip(lower=0))
ridge_log.fit(X_train, y_train_log)
pred_log_val = np.expm1(ridge_log.predict(X_val))
rmse_log = mean_squared_error(y_val, pred_log_val, squared=False)

if rmse_log < rmse_raw:
    chosen_model = ridge_log
    use_log = True
    best_rmse = rmse_log
else:
    chosen_model = ridge_raw
    use_log = False
    best_rmse = rmse_raw

print(f"Validation RMSE (raw): {rmse_raw:.4f}")
print(f"Validation RMSE (log): {rmse_log:.4f}")
print(
    f"Chosen model uses {'log‑target' if use_log else 'raw'} with RMSE: {best_rmse:.4f}"
)




Validation RMSE (raw): 9.6058
Validation RMSE (log): 9.8151
Chosen model uses raw with RMSE: 9.6058


In [22]:
if use_log:
    y_full = np.log1p(y.clip(lower=0))
else:
    y_full = y

chosen_model.fit(X_array, y_full)




Ridge(alpha=0.0001, max_iter=1000, random_state=80, solver='sag', tol=0.001)

In [23]:
test_pred = chosen_model.predict(X_test_array)
if use_log:
    test_pred = np.expm1(test_pred)

test_pred = np.clip(test_pred, 0, 300)
test_pred = np.round(test_pred, 2)




In [24]:
Submission = pd.DataFrame({"key": test_data["key"], "fare_amount": test_pred})
Submission = Submission[["key", "fare_amount"]]




In [25]:
output_path = Path("/kaggle/working/Submission.csv")
Submission.to_csv(output_path, index=False)
print(f"Submission written to {output_path}")

Submission written to /kaggle/working/Submission.csv
